# Completeness Cleaning Notebook

This notebook focuses strictly on **data completeness issues** (missing or empty values).

### Scope:
- Standardize missing values
- Handle incomplete fields:
  - Title
  - Email
  - Amount
  - Dog Gender
  - Dog Age
- Remove noise columns
- Ensure NO missing values remain

> Note: Consistency fixes (e.g., dog_size normalization) are NOT the focus here.


## Step 1: Install Dependencies

In [1]:
%pip install pandas numpy

  Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl (12.4 MB)
Using cached tzdata-2026.1-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3

## Step 2: Import Libraries

In [22]:
import pandas as pd
import numpy as np

## Step 3: Load Dataset

In [23]:
df = pd.read_csv('../../data/raw/dog_survey.csv')
df.columns = df.columns.str.strip().str.lower()
df.head()

,id,title,first_name,last_name,email,amount_spent_on_dog_food,dog_size,dog_gender,dog_age,unnamed: 9,unnamed: 10
0,1,Mrs,Frasquito,Dene,fdene0@dell.com,£80.05,S,M,53,NaN,NaN
1,2,Ms,Lawrence,Kardos,lkardos1@diigo.com,£66.30,XL,M,2,NaN,NaN
2,3,Rev,Lanna,Wintersgill,lwintersgill2@domainmarket.com,£25.84,L,F,71,NaN,NaN
3,4,Rev,Richmound,Kimmitt,rkimmitt3@jiathis.com,£50.76,M,M,9,NaN,NaN
4,5,Rev,Valeda,Dallender,vdallender4@theglobeandmail.com,£83.41,XL,F,23,NaN,NaN


## Step 4: BEFORE Cleaning - Missing Values

In [24]:
missing_before = df.isna().sum()
missing_before

id                            0
title                        30
first_name                    0
last_name                     0
email                         3
amount_spent_on_dog_food     14
dog_size                      2
dog_gender                    7
dog_age                       1
unnamed: 9                  308
unnamed: 10                 308
dtype: int64

## Step 5: Remove Noise Columns

In [25]:
df = df.loc[:, ~df.columns.str.match(r'^unnamed')]

## Step 6: Normalize Missing Values

In [26]:
df = df.replace(
    regex=[
        r'^\s*$',
        r'^-',
        r'^none$', r'^None$', r'^NONE$',
        r'^na$', r'^NA$', r'^n/a$', r'^N/A$',
        r'^nan$', r'^NaN$', r'^NULL$', r'^null$'
    ],
    value=np.nan
)

df.isna().sum()

id                           0
title                       30
first_name                   0
last_name                    0
email                        4
amount_spent_on_dog_food    17
dog_size                     5
dog_gender                   8
dog_age                      1
dtype: int64

## Step 7: Clean Title

In [27]:
valid_titles = {'Mr', 'Mrs', 'Ms', 'Dr', 'Rev', 'Honorable'}

s = df['title'].astype('string').str.strip()
s = s.where(s.isin(valid_titles), pd.NA)

df['title'] = s.fillna('Unknown')

## Step 8: Clean Email

In [28]:
df['email'] = df['email'].astype(str).str.strip()

mask = ~df['email'].str.contains(r'^[^@]+@[^@]+\.[^@]+$', na=False)
df.loc[mask, 'email'] = np.nan

df['email'] = df['email'].fillna('Unknown')

## Step 9: Clean Amount

In [32]:
s = df['amount_spent_on_dog_food'].astype(str)

s = (
    s.str.replace('£', '', regex=False)
     .str.replace(',', '', regex=False)
     .str.strip()
)

df['amount_spent_on_dog_food'] = pd.to_numeric(s, errors='coerce')

df.loc[(df['amount_spent_on_dog_food'] <= 0), 'amount_spent_on_dog_food'] = np.nan

df['amount_spent_on_dog_food'] = df['amount_spent_on_dog_food'].fillna(
    df['amount_spent_on_dog_food'].median()
)

## Step 10: Clean Dog Gender

In [33]:
df['dog_gender'] = df['dog_gender'].fillna('Unknown')

## Step 11: Clean Dog Age

In [34]:
df['dog_age'] = (
    df['dog_age']
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
)

df['dog_age'] = pd.to_numeric(df['dog_age'], errors='coerce')

df.loc[df['dog_age'] <= 0, 'dog_age'] = np.nan

df['dog_age'] = df['dog_age'].fillna(df['dog_age'].median())

## Step 12: Handle Remaining Missing (Dog Size)

Even though this belongs to consistency, we ensure completeness here.

In [35]:
df['dog_size'] = df['dog_size'].fillna('Unknown')

## Step 13: AFTER Cleaning - Missing Values

In [36]:
missing_after = df.isna().sum()
missing_after

id                          0
title                       0
first_name                  0
last_name                   0
email                       0
amount_spent_on_dog_food    0
dog_size                    0
dog_gender                  0
dog_age                     0
dtype: int64

## Step 14: Completeness Improvement

In [37]:
comparison = pd.DataFrame({
    'Before': missing_before,
    'After': missing_after
})

comparison

,Before,After
amount_spent_on_dog_food,14,0.0
dog_age,1,0.0
dog_gender,7,0.0
dog_size,2,0.0
email,3,0.0
first_name,0,0.0
id,0,0.0
last_name,0,0.0
title,30,0.0
unnamed: 10,308,NaN


## Step 15: Export Cleaned Dataset

In [38]:
output_path = '../../data/processed/completeness_dog_survey_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: ../../data/processed/completeness_dog_survey_cleaned.csv
